FILE 7: `CheatCode.py` - THE GOLD STANDARD STRATEGY

   IMPORTANCE: This shows the OPTIMAL motuion strategy. Study this like a 
   playbook.


KEY METHODS EXPLAINED
   
Lines 45-70 - `_wait_for_tf()` -- Blocking TF Lookup
   ...

   Polls the TF buffer every 100ms until the frame appears or timeout. Ground
   truth frames take a moment to publish after simulation starts.
   

LINES 72-185 - `calc_gripper_pose()` - THE CORE MATH
   This is the most important method. It answers: "Given where the port is and
   where the plug is, where shoudl the gripper be?"

   ...

   THE QUATERNION LOGIC: To align the plug with the port, compute the rotation
   difference between them. `q_diff = q_port * q_plug_inverse`. Then apply this
   rotation to the gripper: 

   ...

   SLERP (Spherical Linear Interpolation) smoothly rotates from current to 
   target orientation. `slerp_fraction` goes from 0.0 (current) to 1.0 (aligned)

   ...

   The plug tip and gripper TCP are NOT the same point. This offset accounts for
   the cable hanging from the gripper -- the plug is below/beside the TCP.

   ...

   PI CONTROLLER for xy alignment. The plug doesn't perfectly follow the gripper
   (cable flexibility), so a proportional-integral loop corrects the drift.
   The integral is clipped at +-0.05m to prevent widnup

   ...



LINES 187-258 -- `insert_cable()` -- THE TWO-PHASE STRATEGY

PHASE 1: Approach (Lines 221-236)

```Python

for t in range(0, 100):
    interp_fraction = t / 100.0
   self.set_pose_target(
      move_robot=move_robot,
      pose=self.calc_gripper_pose(
          port_transform,
          slerp_fraction=interp_function,        # Gradually align rotation
          position_fraction=interp_fraction,     # Gradually move to above port
          z_offset=0.2,                          # Stay 20cm above
          reset_xy_integrator=True,              # Don't accumulate error yet   
      )
   ),
   self.sleep_for(0.05)       # 50ms per step
```
   Slowly lower the plug into the port at 0.05 mm per 50ms = 10mm/s. The PI
   controller on xy keeps the plug centered. Descent until 15mm past the port
   surface, then hold for 5 seconds. 


WHAT YOU'LL MODIFY
   This is your REFERENCE IMPLEMENTATION TO BEAT. It uses ground truth 
   (forbidden in eval). Your job is to replicate this strategy using 
   CAMERA-BASED PERCEPTION instead of TF lookups:

   1. Replace `lookup_transform()` calls with a vision model that estimates 
      port/plug poses from the 3 cameras
   2. Keep the same two-phase approach-then-insert strategy
   3. Keep the PI controller for insertion
   4. Tune stiffness/damping for better scoring



---
FILE 8: `RunACT.py` -- The Neural Network Approach
   
   PATH: ...
   IMPORTANCE: Shows how to integrate a learned policy. This is likely closer to
   what your submission will be.

KEY METHODS EXPLAINED

Lines 54-133 -- `__init__()` -- MODEL LOADING && NORMALISATION
   
   ...

   Downloads a pre-trained ACT model from HuggingFace. In your submission, 
   you'll either bundle weights inside the Docker image or download at startup.

   ...

   Standard PyTorch model loading. `.eval()` disables dropout/batch norm 
   training behavior.


NORMALISATION STATS (lines 92-128): The model was trained on normalised data. 
You must apply the same normalisation at inference:
   - IMAGES: per-camera mean/std (shape [1, 3, 1, 1] for broadcasting)
   - State: 26-dimensional mean/std
   - Actions: 7-dimensional mean/std for un-normalisation


LINES 135-167 - `_img_to_tensor()` -- Image Preprocessing Pipeline
   ...
   ROS image arrive as flat byte arrays in HWC format. This converts to
   normalised CHW tensors. Scale factor 0.25 reduces 480x640 (line 131) to
   120x160.



LINES 169-235 -- `prepare_observations()` -- STATE VECTOR CONSTRUCTION
   ...
   This 26-dimensional state vector must EXACTLY MATCH TRAINING ORDER. If you
   train your own model, document this ordering carefully. 



LINES 237-295 -- `insert_cable()` -- THE INFERENCE LOOP
   ...
   The model outputs 7 values: 6 velocity components (linear xyz + angular xyz)
   + 1 (unused gripper). It uses MODE_VELOCITY instead of MODE_POSITION -- the 
   model commands how fast to move, not where to go.


LINES 297-320 -- `set_cartesian_twist_target()` -- VELOCITY CONTROL
   ...
   Unlike `set_pose_target()` which uses MODE_POSITION, the neural network uses
   velocity mode. This is a key design choice -- velocity control is often 
   easier for neural networks to learn because it's relative, not absolute. 



WHAT YOU'LL MODIFY
   This is your TEMPLATE FOR A LEARNED POLICY. You'll:
      1. Train your own ACT/diffusion policy using demonstration (collect via
         teleopration or from CheatCode runs with ground truth)
      2. Potentially switch to a different model architecture (diffusion policy,
         VQ-BeT, etc.)
      3. Adjust the state vector dimensions if you add/remove features
      4. Tune the control frequency (4 Hz here, could go up to 20 Hz)
      5. Bundle your trained weights in the Docker image



---
FILE 9: `WaveArm.py` -- THE MINIMAL WORKING EXAMPLE
   Path: ...
   IMPORTANCE: The simplest possible policy. Start here to verify your setup 
               works.

   KEY LINES EXPLAINED

   ...
   Uses the IMAGE TIMESTAMP for time-based animation, not wall clock. This is 
   sim-time-aware.













---

   You have a ... caught a glaring architectural discrepancy in the provided
   codebase, and it highlights one of the most common pitfalls when bridging
   Deep Learning with Robotics.

   You are completely correct: `WaveArm.py` and `CheatCode.py` rigorously use 
   `self.time_now()` and `self.sleep_for()`, while `RunACT.py` abandons them
   entirely in favor of Python's standard `time.time()` and `time.sleep()`.

   ... why that happens, and why it is actually a "direty secret" of that 
   specific proof-of-concept script.


1. `WaveArm` && `CheatCode`


















-- HWC format refers to storing image data in Height, Width, and Channel order
   (e.g., [1080, 1920]), commonly used in computer vision libraries like TF for
   pixel-wide access. It implies pixels are stored contiguously, such as 
   R, G, B, R, G, B, where the color channel (C) varies fastest.

The confusion here is completely understyandable, and it comes down to a quirk
in programming terminology! When you read "Blocking TF Lookup" in the context of
`CheatCode.py`, the word "blocking" does not mean "preventing" or "banning".

In programming, a. "BLOCKING FUNCTION" simply means it pauses or halts the
execution of the rest of the code until a specific coondition is met. In the
`_wait_for_tf` enters a `while` loop that continuously asks the `tf_buffer`, "Is
the coordinate here yet? How about now? Now?..." It literally blocks the robot
rom moving onto the next line of code until the exact coordinates of the port
appears in the buffer (or until it times out after 10 seconds).

As for GROUND TRUTH, you are exactly right about its availability. Ground Truth
is the absolute, mathematically perfect coordinate of where an object (like the
NIC card or SFP port) exists in the 3D simulation space. When 
`ground_truth:=true` is enabled during your local development, the Gazebo 
physics engine bypasses the robot's cameras entirely and directly injects these
perfect coordinates intot he robot's `tf_buffer`. It is basically giving the 
robot a magical GPS that tells it exactly where the target is, down to the
millimeter. 

You nailed it: GROUND TRUTH IS STRICTLY FOR TRAINING AND DEBUGGING, and it is
physically impossible in reality. In the real world, a server rack doesn't 
broadcast its exact coordinates to a robot's brain. During the official 
Evaluation phase, the organisers will set `ground_truth:=false`. This cuts off
the magical GPS feed. If your policy tries to run that `wait_for_tf` loop during
evaluation, the `tf_buffer` will never receive the port's coordinates, the loop
will time out, and your robot will stand there and fail the trial.

The `CheatCode.py` file is provided purely as a "Gold Standard." It uses that
magical Ground Truth to show you what perfect motion looks like. Your ultimate
goal is to train an AI (like the ACT model) to mimic that exact perfect motion,
but by using only camera images to figure out where to go, instead of relying
on those banned TF coordinates!